# [LAB] MCP 서버와 클라이언트 실습

MCP 서버를 만들어 터미널에 띄우고, 노트북의 Agent가 그 서버에 연결해 도구를 쓰게 합니다. 쉬운 기본 예제로 구조를 익힌 뒤, 실제 데이터를 쓰는 실전 프로젝트로 넘어갑니다.

- **2장 기본: MCP 서버 만들고 띄우기**: 현재 시간·BMI·할 일 도구가 든 서버 코드를 셀로 나눠 읽고, 터미널에서 띄웁니다(`basic_mcp_server.py`, 포트 8000).
- **3장 기본: 클라이언트로 연결하기**: 서버에 연결해 도구를 가져오고 Agent 그래프를 만듭니다. 코드는 완성되어 있습니다. 실행해 본 뒤 연결 방식을 SSE에서 stdio로 바꿔 비교합니다.
- **4장 실전 프로젝트: DART 공시 MCP 서버**: 금융감독원 전자공시(DART) API를 도구 세 개로 감싼 서버를 띄웁니다(`dart_mcp_server.py`, 포트 8002).
- **5장 실전 프로젝트: DART와 Tavily를 한 Agent에**: 우리 DART 서버와 Tavily가 운영하는 웹 검색 MCP 서버에 함께 연결해 공시·실적·뉴스 질문에 답하게 합니다. 코드는 완성되어 있습니다.

**시작 전에: API 키 두 개 넣기**

- **DART**: 강사가 알려 주는 40자리 인증키를 `.env`의 `DART_API_KEY=` 뒤에 붙여 넣습니다.
- **Tavily** (각자, 무료): `app.tavily.com`에 가입하면 첫 화면에 `tvly-`로 시작하는 키가 있습니다. `.env`의 `TAVILY_API_KEY=` 뒤에 붙여 넣습니다.
- `.env`를 고쳤으면 커널을 다시 시작합니다. 4장의 DART 서버를 이미 띄웠다면 서버도 Ctrl + C로 끄고 다시 띄웁니다.
- 키가 없거나 사내망에서 연결되지 않아도 실습은 끝까지 됩니다. DART 도구는 `[모의 DART]`로 시작하는 연습용 답을 돌려주고, Tavily는 빼고 진행합니다.

키는 4·5장에서만 씁니다. 셀은 위에서부터 차례로 실행합니다. `# ✍️ CHANGE_HERE:`가 있는 셀은 먼저 그대로 실행한 뒤, 셀 위의 **바꿔 볼 값**으로 바꿔 다시 실행해 봅니다. 3장은 2장의 서버, 5장은 4장의 서버가 터미널에 켜져 있어야 동작합니다(두 서버는 포트가 달라 함께 켜 두어도 됩니다).

**목차**

- 1. 실습 방법
- 2. 기본: MCP 서버 만들고 띄우기
  - 2.1 서버 만들기
  - 2.2 Tool 정의하기
  - 2.3 Resource 정의하기
  - 2.4 Prompt 정의하기
  - 2.5 실행 방식 정하기
- 3. 기본: 클라이언트로 연결하기
  - 3.1 연결 방식 정하기
  - 3.2 클라이언트 만들기
  - 3.3 도구 목록 가져오기
  - 3.4 LLM과 도구 노드에 도구 연결하기
  - 3.5 agent 노드
  - 3.6 그래프 연결과 컴파일
  - 3.7 질문하기
- 4. 실전 프로젝트: DART 공시 MCP 서버 띄우기
  - 4.1 서버 만들기
  - 4.2 DART API 호출하기
  - 4.3 Tool 정의하기
  - 4.4 Resource 정의하기
  - 4.5 Prompt 정의하기
  - 4.6 실행 방식 정하기
- 5. 실전 프로젝트: DART와 Tavily를 한 Agent에 연결하기
  - 5.1 연결할 서버 정하기
  - 5.2 도구 가져오기
  - 5.3 Agent 그래프 만들기
  - 5.4 질문하기
- 6. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env(["MCP_KEYS"])

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고칩니다. `.env`는 노트북을 시작할 때 한 번만 읽으므로, 고친 뒤에는 커널을 다시 시작하고 맨 위 셀부터 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

도구를 부를 때 `Error in ConsoleCallbackHandler.on_tool_start callback: KeyError('input')` 같은 줄이 섞여 보일 수 있습니다. 디버그 내용을 출력하는 쪽의 문제라 실행 결과에는 영향이 없으니 그대로 두면 됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

## 1. 실습 방법

코드는 모두 완성되어 있습니다. 위에서부터 차례로 실행합니다.

- `# ✍️ CHANGE_HERE:` 주석이 있는 셀은 먼저 그대로 실행해 결과를 봅니다. 그다음 셀 위의 **바꿔 볼 값**을 복사해, 주석 바로 아래 줄의 표시된 자리(지금 값)를 지우고 붙여 넣습니다. 어떻게 달라질지 생각해 본 뒤 다시 실행합니다.
- 달라진 결과를 본 뒤 셀 아래 접힌 **무슨 일이 생기나**를 펼쳐 비교합니다. 확인했으면 값을 원래대로 되돌립니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. 기본: MCP 서버 만들고 띄우기

FastMCP로 MCP 서버를 만듭니다. 서버는 노트북 셀 안에서 돌리지 않고 터미널에서 띄웁니다. 서버는 켜 둔 동안 계속 요청을 기다리기 때문에, 노트북 셀에서 돌리면 다른 셀을 실행할 수 없습니다.

이 폴더에는 완성된 서버 파일 `basic_mcp_server.py`가 미리 들어 있습니다. 아래 셀은 그 파일의 코드를 나눠 보여 주는 곳이라 차례로 읽고 실행해 봅니다. 셀을 실행해도 서버 파일은 바뀌지 않습니다.

서버에는 3대 Primitive가 모두 들어 있습니다.

- **Tool 5개:** 현재 시간, BMI 계산, 할 일 추가·목록·완료
- **Resource 1개:** 서버 정보(`config://app-info`)
- **Prompt 1개:** 할 일 요약 요청 템플릿

셀을 다 본 뒤, 아래 안내대로 터미널에서 서버를 띄웁니다.

### 2.1 서버 만들기

In [ ]:
"""
MCP 서버 만들기 (FastMCP)

터미널에서 실행한다. 기본은 HTTP/SSE 모드(포트 8000), --stdio를 붙이면 stdio 모드로 실행한다.
"""

import sys
from datetime import datetime

from fastmcp import FastMCP

# FastMCP: Anthropic MCP SDK 위에서 동작하는 고수준 래퍼.
# @mcp.tool(), @mcp.resource(), @mcp.prompt() 데코레이터로 기능을 정의한다.
mcp = FastMCP("AI 강의 튜토리얼 서버")

### 2.2 Tool 정의하기

In [ ]:
# ── Tool: LLM이 호출할 수 있는 함수 ──────────────────────────────────────────
@mcp.tool()
def get_current_time() -> str:
    """현재 날짜와 시간을 반환한다."""
    return datetime.now().strftime("%Y년 %m월 %d일 %H:%M:%S")


@mcp.tool()
def calculate_bmi(weight_kg: float, height_cm: float) -> dict:
    """체중(kg)과 키(cm)로 BMI를 계산한다."""
    height_m = height_cm / 100
    bmi = weight_kg / (height_m**2)
    if bmi < 18.5:
        category = "저체중"
    elif bmi < 23:
        category = "정상"
    elif bmi < 25:
        category = "과체중"
    else:
        category = "비만"
    return {"bmi": round(bmi, 1), "category": category}


_tasks: list[dict] = []


@mcp.tool()
def add_task(title: str, priority: int = 1) -> str:
    """할 일 목록에 새 항목을 추가한다. priority는 1(낮음)~3(높음)."""
    task = {"id": len(_tasks) + 1, "title": title, "priority": priority, "done": False}
    _tasks.append(task)
    return f"할 일 추가됨: [{task['id']}] {title} (우선순위: {priority})"


@mcp.tool()
def list_tasks() -> list[dict]:
    """현재 할 일 목록을 반환한다."""
    return _tasks if _tasks else [{"message": "할 일이 없습니다."}]


@mcp.tool()
def complete_task(task_id: int) -> str:
    """할 일을 완료 처리한다."""
    for task in _tasks:
        if task["id"] == task_id:
            task["done"] = True
            return f"완료: {task['title']}"
    return f"ID {task_id}를 찾을 수 없습니다."

### 2.3 Resource 정의하기

In [ ]:
# ── Resource: LLM에게 제공할 데이터/문서 ──────────────────────────────────────
@mcp.resource("config://app-info")
def get_app_info() -> str:
    """이 MCP 서버의 기본 정보를 제공한다."""
    return """
    서버명: AI 강의 튜토리얼 MCP 서버
    버전: 1.0.0
    기능: 시간 조회, BMI 계산, 할 일 관리
    """

### 2.4 Prompt 정의하기

In [ ]:
# ── Prompt: 자주 쓰는 프롬프트 템플릿 ────────────────────────────────────────
@mcp.prompt()
def task_summary_prompt(user_name: str) -> str:
    """할 일 목록 요약 요청 프롬프트를 생성한다."""
    return f"{user_name}님의 할 일 목록을 우선순위별로 정리하고 오늘 집중해야 할 항목을 추천해줘."

### 2.5 실행 방식 정하기

아래는 서버를 터미널에서 띄울 때만 실행되는 부분입니다. 노트북에서는 실행하지 않고 읽어만 봅니다.

```python
if __name__ == "__main__":
    if "--stdio" in sys.argv:
        # stdio 모드: Claude Code, Cursor 등 MCP Host와 직접 연결
        mcp.run(transport="stdio")
    else:
        print("MCP 서버 시작: http://localhost:8000/sse")
        print("종료하려면 Ctrl+C")
        # HTTP/SSE 모드: 클라이언트가 localhost:8000으로 연결
        # 127.0.0.1: 이 PC에서 오는 요청만 받는다. 0.0.0.0이면 다른 PC의 요청도 받는다.
        mcp.run(transport="sse", host="127.0.0.1", port=8000)
```

이 폴더에는 완성된 서버 파일(`basic_mcp_server.py`)이 미리 들어 있습니다. 2.1부터 위 셀들은 그 파일의 코드를 나눠 보여 주는 곳입니다. 셀을 실행해 오류가 없는지 확인합니다. 셀이 파일을 고치지는 않습니다.

서버는 터미널에서 이 완성 파일로 띄웁니다.

1. VS Code 메뉴 **터미널 → 새 터미널**을 엽니다(저장소 맨 위 폴더에서 열립니다).
2. 아래 명령을 붙여 넣고 실행합니다.

```
uv run python course/Chapter02/notebooks/basic_mcp_server.py
```

3. 서버 시작 문구가 나오고 명령이 끝나지 않고 기다리면 켜진 것입니다. 이 터미널은 닫지 말고 노트북으로 돌아옵니다. 서버를 끌 때는 터미널에서 **Ctrl + C**를 누릅니다.

**터미널 화면**

- `MCP 서버 시작: http://localhost:8000/sse`와 `종료하려면 Ctrl+C` 두 줄이 보인다
- 그 아래 FastMCP 안내 상자(서버 이름 `AI 강의 튜토리얼 서버`가 적힌 상자)가 나오고, 맨 끝 줄에 `Uvicorn running on`과 `(Press CTRL+C to quit)`가 보인다
- 명령이 끝나지 않고 기다린다(새 명령을 입력하는 줄이 돌아오지 않는다)
- 안 되면: 명령이 곧바로 끝나고 오류가 나온다 → 오류 메시지 전체를 AI에게 보여 줍니다
- 안 되면: 오류에 `address already in use` 또는 `10048`이 보인다 → 이미 켜 둔 서버가 있습니다. 앞서 서버를 띄운 터미널에서 Ctrl+C로 끄고 다시 띄웁니다

**볼 지점**
- 서버를 띄운 터미널은 3장이 끝날 때까지 켜 둡니다. 3장에서 도구를 호출할 때마다 이 터미널에 요청 기록(`POST /messages/...` 같은 줄)이 늘어납니다.
- 안내 상자에 새 버전 안내(`Update available`)가 함께 나올 수 있습니다. 실습과 관계없으니 그대로 둡니다.

## 3. 기본: 클라이언트로 연결하기

노트북에서 Agent(Host)를 만들고, 그 안의 Client로 2장의 서버에 연결합니다. 서버의 도구를 가져와 LLM에 넘기고, 과정 1에서 만든 것과 같은 모양의 LangGraph 그래프(agent ⇄ tools)로 질문에 답하게 합니다.

- **연결:** 클라이언트 만들기, 도구 목록 가져오기
- **Agent 그래프:** LLM과 도구 노드에 도구 연결하기, agent 노드, 조건부 엣지, 컴파일

서버와 주고받는 코드 앞에는 `await`가 붙습니다. 응답이 올 때까지 기다린다는 표시입니다. 노트북 셀에서는 `await`를 바로 쓸 수 있습니다.

**왜 배우나:** 도구를 서버에서 가져온다는 점만 다르고, 나머지는 과정 1의 `bind_tools`·ToolNode·조건부 엣지를 그대로 씁니다. 도구가 어디에 있든 Agent 쪽 코드는 같다는 것을 직접 확인합니다.

준비 코드입니다. 필요한 도구를 불러오고 설정 파일을 읽습니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다.

In [ ]:
"""
MCP 클라이언트 실습: 터미널에 켜 둔 MCP 서버에 연결해 LangGraph Agent가 도구로 사용한다.
"""

import sys
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm
from config.mcp import stdio_errors_to_file

load_dotenv();

### 3.1 연결 방식 정하기

**✍️ CHANGE_HERE: 연결 방식을 "stdio"로 바꾸면 3.7의 할 일 질문 답이 어떻게 달라질까요?**

**바꿔 볼 값** — 바로 아래 줄의 `"sse"` 자리에 복사해 붙여 넣기

```text
"stdio"
```

먼저 3장 끝까지 그대로 실행해 봅니다. 값을 바꿨으면 이 셀부터 3.7 질문하기 셀까지 차례로 다시 실행합니다. 확인했으면 `"sse"`로 되돌리고 같은 셀들을 다시 실행합니다.

In [ ]:
# ── MCP 서버 연결 설정 ────────────────────────────────────────────────────────
# SSE(Server-Sent Events): HTTP 기반 MCP 연결 방식.
# 서버를 HTTP 모드로 실행하면 /sse 엔드포인트가 열린다. 터미널에 켜 둔 서버에 주소로 접속한다.
# stdio: 노트북이 서버 파일을 직접 실행해 표준 입출력으로 주고받는다.
# stdio로 띄운 서버의 오류 출력은 화면 대신 mcp_stdio_server.log 파일로 받는다
# (Windows 노트북의 화면 출력은 진짜 파일이 아니라, 그대로 두면 서버를 띄우지 못한다)
stdio_errors_to_file("mcp_stdio_server.log")
CONNECTIONS = {
    "sse": {"transport": "sse", "url": "http://localhost:8000/sse"},
    "stdio": {
        "transport": "stdio",
        "command": sys.executable,
        "args": ["basic_mcp_server.py", "--stdio"],
    },
}
# ✍️ CHANGE_HERE: 연결 방식을 "stdio"로 바꾸면 3.7의 할 일 질문 답이 어떻게 달라질까요?
TRANSPORT = "sse"
MCP_CONFIG = {"tutorial": CONNECTIONS[TRANSPORT]}

<details><summary>무슨 일이 생기나</summary>

도구 목록은 그대로 5개이고, 시간·BMI 질문의 답도 SSE 때와 같습니다. 달라지는 것은 할 일 질문입니다. `add_task`의 Tool Message는 `할 일 추가됨: [1] LangGraph 공부`인데, 함께 부른 `list_tasks`의 Tool Message는 `할 일이 없습니다.`입니다(2026-10-21 사외 실행).

stdio에서는 노트북이 도구를 부를 때마다 서버 프로그램을 새로 띄우고 끝냅니다. 할 일을 추가한 서버와 목록을 보여 준 서버가 서로 다른 서버라 목록이 비어 있습니다. 이번 실행의 마지막 답은 그런데도 목록에 `LangGraph 공부`가 있다고 말했습니다. 모델의 답이 도구 결과와 다를 수 있으니 Tool Message를 직접 봅니다.

터미널의 서버에는 새 요청 기록이 생기지 않고, stdio로 띄운 서버의 기록은 노트북 폴더의 `mcp_stdio_server.log`에 쌓입니다.

</details>

### 3.2 클라이언트 만들기

위 셀에서 만든 연결 설정 `MCP_CONFIG`로 클라이언트를 만들어 변수 `client`에 담는 한 줄입니다. 이 줄은 서버에 바로 접속하지 않습니다. 설정만 들고 있다가 다음 셀에서 도구를 가져올 때 접속합니다.

In [ ]:
client = MultiServerMCPClient(MCP_CONFIG)

### 3.3 도구 목록 가져오기

서버에 접속해 도구 이름과 설명을 읽어 LangChain Tool 목록으로 받아 `tools`에 담는 한 줄입니다. 서버와 주고받는 동안 기다려야 하므로 `await`가 필요합니다. 아래 줄들이 받은 도구를 하나씩 출력합니다.

In [ ]:
# get_tools()는 서버에 연결해 도구 목록을 가져온다.
tools = await client.get_tools()
print("=== 사용 가능한 MCP 도구 ===")
for t in tools:
    print(f"  - {t.name}: {t.description}")
print()

### 3.4 LLM과 도구 노드에 도구 연결하기

두 줄입니다. 하나는 `get_llm()`으로 만든 모델에 서버의 도구 목록을 알려 주어 `llm`에 담고, 다른 하나는 같은 도구 목록으로 도구를 실제로 실행하는 노드 `tool_node`를 만듭니다. 과정 1의 Tool Calling과 LangGraph에서 쓴 방법과 같습니다.

In [ ]:
llm = get_llm().bind_tools(tools)
tool_node = ToolNode(tools)

### 3.5 agent 노드

`agent_node`는 Agent가 한 번 생각하는 단계입니다. 두 줄입니다. 상태에 쌓인 메시지를 도구가 연결된 `llm`에 보내 응답을 받고, 그 응답을 `messages`에 더하도록 딕셔너리로 돌려줍니다.

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]


def agent_node(state: State) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

### 3.6 그래프 연결과 컴파일

두 줄입니다. agent 노드 다음에 LLM이 도구 호출을 요청했으면 tools로, 아니면 끝(END)으로 가게 조건부 엣지를 답니다. 그리고 tools 노드가 도구를 실행한 뒤에는 다시 agent로 돌아와 결과를 보고 답하게 합니다.

노드와 엣지를 다 붙인 `builder`를 실행할 수 있는 그래프로 만들어 `app`에 담는 한 줄입니다. 다음 셀이 `app.ainvoke(...)`로 이 그래프를 실행합니다.

In [ ]:
builder = StateGraph(State)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

app = builder.compile()

### 3.7 질문하기

**✍️ CHANGE_HERE: 첫 질문을 도구가 필요 없는 질문으로 바꾸면 그래프가 어디를 지나갈까요?**

**바꿔 볼 값** — 바로 아래 줄의 `지금 몇 시야?` 자리에 복사해 붙여 넣기

```text
MCP가 뭐야?
```

이 셀을 다시 실행하고 첫 질문의 메시지 칸을 봅니다. 확인했으면 `지금 몇 시야?`로 되돌립니다.

In [ ]:
# 도구를 사용해 요청 처리
questions = [
    # ✍️ CHANGE_HERE: 첫 질문을 도구가 필요 없는 질문으로 바꾸면 그래프가 어디를 지나갈까요?
    "지금 몇 시야?",
    "체중 70kg, 키 175cm면 BMI가 얼마야?",
    '할 일 목록에 "LangGraph 공부" (우선순위 3)를 추가하고 현재 목록을 보여줘.',
]

for question in questions:
    mcp_graph_result = await app.ainvoke(
        {
            "messages": [HumanMessage(content=question)],
        }
    )
    results = mcp_graph_result["messages"]
    for m in results:
        m.pretty_print()
    print()

<details><summary>무슨 일이 생기나</summary>

첫 질문은 `Human Message` → `Ai Message` 두 칸으로 끝납니다. `Tool Calls`와 `Tool Message` 칸이 없습니다. LLM이 도구가 필요 없다고 보고 도구 호출 없이 답했고, 조건부 엣지(`tools_condition`)가 tools 노드를 거치지 않고 끝으로 보냈습니다. 나머지 두 질문은 그대로 도구를 부릅니다.

이번 실행의 답은 `MCP는 문맥 상 특정한 의미를 가지지 않으며…`처럼 엉뚱했습니다(2026-10-21 사외 실행). 도구 없이 모델이 아는 것만으로 답하면 이렇게 틀릴 수 있습니다.

</details>

**볼 지점** (질문하기 셀)
- 질문마다 `Human Message` → `Ai Message`(도구 호출) → `Tool Message`(서버 도구의 결과) → `Ai Message`(답) 칸이 나옵니다. 세 질문 모두 LLM이 서버의 도구를 호출해 얻은 값으로 답합니다.
  - 시간 질문: 지금 날짜와 시각
  - BMI 질문: BMI 22.9, 정상 범위
  - 할 일 질문: "LangGraph 공부"(우선순위 3)가 든 목록
- 서버 터미널에 도구 호출마다 요청 기록이 늘어납니다. Agent는 노트북에서, 도구는 터미널의 서버에서 실행된다는 뜻입니다.
- 서버를 켠 채로 질문하기 셀을 다시 실행하면 할 일 목록에 "LangGraph 공부"가 하나씩 더 쌓입니다. 서버가 켜져 있는 동안 목록을 기억하기 때문이며 정상입니다. 서버를 껐다 켜면 비워집니다.
- 답의 말투와 모양은 실행할 때마다 조금씩 다릅니다.

## 4. 실전 프로젝트: DART 공시 MCP 서버 띄우기

2·3장에서 익힌 그대로, 이번에는 실제 데이터를 주는 MCP 서버를 씁니다. 금융감독원 전자공시(DART)의 공개 API를 MCP 도구로 감싼 서버를 만듭니다. DART는 상장회사가 낸 공시와 사업보고서를 누구나 볼 수 있게 내놓는 곳입니다. 서버는 노트북 셀 안에서 돌리지 않고 터미널에서 띄웁니다. 서버는 켜 둔 동안 계속 요청을 기다리기 때문에, 노트북 셀에서 돌리면 다른 셀을 실행할 수 없습니다.

이 폴더에는 완성된 서버 파일 `dart_mcp_server.py`가 미리 들어 있습니다. 아래 셀은 그 파일의 코드를 나눠 보여 주는 곳이라 차례로 읽고 실행해 봅니다. 셀을 실행해도 서버 파일은 바뀌지 않고, DART도 아직 호출하지 않습니다.

서버에는 3대 Primitive가 모두 들어 있습니다.

- **Tool 3개:** 회사 기본 정보, 최근 공시 목록, 한 해 실적(매출액·영업이익·당기순이익)
- **Resource 1개:** 찾을 수 있는 회사 목록(`dart://companies`)
- **Prompt 1개:** 회사 공시 브리핑 요청 템플릿

`DART_API_KEY`가 없거나 DART에 연결되지 않으면 도구는 `[모의 DART]`로 시작하는 연습용 답을 돌려줍니다. 실제 공시가 아니라는 표시이니 그 답의 숫자는 믿지 않습니다.

셀을 다 본 뒤, 아래 안내대로 터미널에서 서버를 띄웁니다.

### 4.1 서버 만들기

In [ ]:
"""
DART 공시 MCP 서버 만들기 (FastMCP)

금융감독원 전자공시(OpenDART) API를 MCP 도구로 감싼다. 터미널에서 실행한다.
기본은 HTTP/SSE 모드(포트 8002. 기본 실습 서버가 8000을 쓴다), --stdio를 붙이면 stdio 모드로 실행한다.
.env에 DART_API_KEY가 없거나 DART에 연결되지 않으면 [모의 DART]로 시작하는 연습용 답을 돌려준다.
"""

import os
import sys
from datetime import date, timedelta

import requests
import truststore
from dotenv import load_dotenv
from fastmcp import FastMCP

truststore.inject_into_ssl()  # 사내망 HTTPS는 회사 인증서를 거친다. PC에 등록된 인증서를 믿게 한다
load_dotenv()

# FastMCP: Anthropic MCP SDK 위에서 동작하는 고수준 래퍼.
# @mcp.tool(), @mcp.resource(), @mcp.prompt() 데코레이터로 기능을 정의한다.
mcp = FastMCP("DART 공시 서버")

### 4.2 DART API 호출하기

In [ ]:
# ── DART API 호출하기 ──────────────────────────────────────────────────────────
DART_URL = "https://opendart.fss.or.kr/api"
# DART는 회사를 8자리 고유번호(corp_code)로 찾는다. 실습에서 쓸 회사만 사전으로 둔다
COMPANIES = {
    "삼성전자": "00126380",
    "SK하이닉스": "00164779",
    "LG전자": "00401731",
    "현대자동차": "00164742",
    "NAVER": "00266961",
    "카카오": "00258801",
}
MOCK = "[모의 DART] DART_API_KEY가 없거나 DART에 연결하지 못해 연습용 예시를 보여 줍니다. 실제 공시가 아닙니다."


def call_dart(endpoint: str, **params) -> dict | None:
    """DART API를 호출한다. 키가 없거나 DART에 연결하지 못하면 None을 돌려준다(그때 도구는 [모의 DART] 답을 낸다)."""
    key = os.getenv("DART_API_KEY", "").strip()
    if not key:
        return None
    try:
        return requests.get(
            f"{DART_URL}/{endpoint}", params={"crtfc_key": key, **params}, timeout=10
        ).json()
    except (requests.RequestException, ValueError):
        return None


def find_code(company: str) -> str | None:
    return COMPANIES.get(company.replace(" ", "").replace("(주)", ""))


def unknown(company: str) -> str:
    return f"'{company}'은(는) 이 서버의 회사 사전에 없습니다. 찾을 수 있는 회사: {', '.join(COMPANIES)}"

### 4.3 Tool 정의하기

In [ ]:
# ── Tool: LLM이 호출할 수 있는 함수 ──────────────────────────────────────────
@mcp.tool()
def get_company_info(company: str) -> str:
    """회사의 기본 정보(정식 이름, 대표이사, 종목코드, 설립일, 홈페이지)를 DART에서 찾는다."""
    code = find_code(company)
    if code is None:
        return unknown(company)
    data = call_dart("company.json", corp_code=code)
    if data is None:
        return f"{MOCK}\n회사명: {company}(예시) / 대표이사: 홍길동 / 종목코드: 000000 / 설립일: 19690101"
    if data.get("status") != "000":
        return f"DART 응답: {data.get('message')}"
    return (
        f"회사명: {data['corp_name']} / 대표이사: {data['ceo_nm']} / 종목코드: {data['stock_code']} / "
        f"설립일: {data['est_dt']} / 홈페이지: {data['hm_url']}"
    )


@mcp.tool()
def get_recent_disclosures(company: str, days: int = 30) -> str:
    """회사가 최근 며칠 동안 DART에 낸 공시 목록(날짜, 공시 제목, 제출인)을 찾는다. 기본은 30일."""
    code = find_code(company)
    if code is None:
        return unknown(company)
    end = date.today()
    begin = end - timedelta(days=days)
    data = call_dart(
        "list.json",
        corp_code=code,
        bgn_de=begin.strftime("%Y%m%d"),
        end_de=end.strftime("%Y%m%d"),
        page_count=10,
    )
    if data is None:
        return f"{MOCK}\n{end} 주요사항보고서(예시) / {begin} 분기보고서(예시)"
    if data.get("status") != "000":
        return f"DART 응답: {data.get('message')}"
    day = lambda d: f"{d[:4]}-{d[4:6]}-{d[6:]}"  # 20261019 → 2026-10-19
    rows = [
        f"{day(x['rcept_dt'])} {x['report_nm'].strip()} (제출: {x['flr_nm']})"
        for x in data["list"]
    ]
    return (
        f"{data['list'][0]['corp_name']} 최근 {days}일 공시 {data['total_count']}건 중 최근 {len(rows)}건:\n"
        + "\n".join(rows)
    )


@mcp.tool()
def get_financial_summary(company: str, year: int) -> str:
    """회사의 한 해 실적(연결 기준 매출액, 영업이익, 당기순이익)을 그해 사업보고서에서 찾는다."""
    code = find_code(company)
    if code is None:
        return unknown(company)
    data = call_dart(
        "fnlttSinglAcnt.json", corp_code=code, bsns_year=str(year), reprt_code="11011"
    )
    if data is None:
        return f"{MOCK}\n{year}년 매출액 100.0조 원 / 영업이익 10.0조 원 / 당기순이익 8.0조 원(예시)"
    if data.get("status") != "000":
        return f"DART 응답: {data.get('message')} ({year}년 사업보고서)"
    lines = []
    for x in data["list"]:
        if x["fs_div"] == "CFS" and x["account_nm"] in (
            "매출액",
            "영업이익",
            "당기순이익(손실)",
            "당기순이익",
        ):
            now, before = (
                int(v.replace(",", "")) / 1e12
                for v in (x["thstrm_amount"], x["frmtrm_amount"])
            )
            lines.append(f"{x['account_nm']}: {now:.1f}조 원 (전년 {before:.1f}조 원)")
    return f"{company} {year}년 연결 실적\n" + "\n".join(
        dict.fromkeys(lines)
    )  # 같은 줄이 두 번 오는 회사가 있다

### 4.4 Resource 정의하기

In [ ]:
# ── Resource: LLM에게 제공할 데이터/문서 ──────────────────────────────────────
@mcp.resource("dart://companies")
def list_companies() -> str:
    """이 서버가 찾을 수 있는 회사와 DART 고유번호 목록."""
    return "\n".join(f"{name}: {code}" for name, code in COMPANIES.items())

### 4.5 Prompt 정의하기

In [ ]:
# ── Prompt: 자주 쓰는 프롬프트 템플릿 ────────────────────────────────────────
@mcp.prompt()
def company_briefing(company: str) -> str:
    """회사 한 곳의 공시 브리핑을 요청하는 프롬프트를 만든다."""
    return f"{company}의 기본 정보, 최근 30일 공시, 지난해 실적을 DART 도구로 확인하고 꼭 알아야 할 점을 세 줄로 정리해줘."

### 4.6 실행 방식 정하기

아래는 서버를 터미널에서 띄울 때만 실행되는 부분입니다. 노트북에서는 실행하지 않고 읽어만 봅니다.

```python
if __name__ == "__main__":
    if "--stdio" in sys.argv:
        # stdio 모드: 클라이언트가 이 파일을 직접 실행해 표준 입출력으로 주고받는다(print를 쓰면 통신이 깨진다)
        mcp.run(transport="stdio")
    else:
        print("DART MCP 서버 시작: http://localhost:8002/sse")
        print(
            "DART_API_KEY:",
            "있음 — 실제 DART를 호출합니다"
            if os.getenv("DART_API_KEY", "").strip()
            else "없음 — [모의 DART] 연습용 답을 돌려줍니다",
        )
        print("종료하려면 Ctrl+C")
        # HTTP/SSE 모드: 클라이언트가 localhost:8002로 연결
        # 127.0.0.1: 이 PC에서 오는 요청만 받는다. 0.0.0.0이면 다른 PC의 요청도 받는다.
        mcp.run(transport="sse", host="127.0.0.1", port=8002)
```

이 폴더에는 완성된 서버 파일(`dart_mcp_server.py`)이 미리 들어 있습니다. 4.1부터 위 셀들은 그 파일의 코드를 나눠 보여 주는 곳입니다. 셀을 실행해 오류가 없는지 확인합니다. 셀이 파일을 고치지는 않습니다.

서버는 터미널에서 이 완성 파일로 띄웁니다.

1. VS Code 메뉴 **터미널 → 새 터미널**을 엽니다(저장소 맨 위 폴더에서 열립니다).
2. 아래 명령을 붙여 넣고 실행합니다.

```
uv run python course/Chapter02/notebooks/dart_mcp_server.py
```

3. 서버 시작 문구가 나오고 명령이 끝나지 않고 기다리면 켜진 것입니다. 이 터미널은 닫지 말고 노트북으로 돌아옵니다. 서버를 끌 때는 터미널에서 **Ctrl + C**를 누릅니다.

**터미널 화면**

- `DART MCP 서버 시작: http://localhost:8002/sse`, `DART_API_KEY: 있음 — 실제 DART를 호출합니다`, `종료하려면 Ctrl+C` 세 줄이 보인다(키가 없으면 둘째 줄이 `DART_API_KEY: 없음 — [모의 DART] 연습용 답을 돌려줍니다`)
- 그 아래 FastMCP 안내 상자(서버 이름 `DART 공시 서버`가 적힌 상자)가 나오고, 맨 끝 줄에 `Uvicorn running on`과 `(Press CTRL+C to quit)`가 보인다
- 명령이 끝나지 않고 기다린다(새 명령을 입력하는 줄이 돌아오지 않는다)
- 안 되면: 명령이 곧바로 끝나고 오류가 나온다 → 오류 메시지 전체를 AI에게 보여 줍니다
- 안 되면: 오류에 `address already in use` 또는 `10048`이 보인다 → 이미 켜 둔 서버가 있습니다. 앞서 서버를 띄운 터미널에서 Ctrl+C로 끄고 다시 띄웁니다

**볼 지점**
- 서버를 띄운 터미널은 5장이 끝날 때까지 켜 둡니다. 5장에서 DART 도구를 호출할 때마다 이 터미널에 요청 기록(`POST /messages/...` 같은 줄)이 늘어납니다.
- 서버는 시작할 때 `.env`를 읽습니다. DART 키를 나중에 넣었다면 이 서버를 Ctrl+C로 끄고 다시 띄웁니다.
- 안내 상자에 새 버전 안내(`Update available`)가 함께 나올 수 있습니다. 실습과 관계없으니 그대로 둡니다.

## 5. 실전 프로젝트: DART와 Tavily를 한 Agent에 연결하기

3장의 Agent와 코드 모양이 같습니다. 다른 점은 연결하는 서버뿐입니다. 하나는 4장에서 띄운 우리 DART 서버이고, 다른 하나는 Tavily가 인터넷에 운영하는 웹 검색 MCP 서버(남이 만든 서버)입니다. 코드는 모두 완성되어 있으니 차례로 실행하며 결과를 봅니다.

**왜 배우나:** 남이 만든 서버든 내가 만든 서버든 연결 설정만 다르고, 받아 온 도구는 똑같이 `bind_tools`·ToolNode·조건부 엣지로 씁니다. MCP가 Agent에 도구를 꽂아 쓰는 표준 단자라는 것을 직접 확인합니다.

준비 코드입니다. 필요한 도구를 불러오고, 설정 파일을 읽고, MCP 서버에 닿는지 확인하는 함수를 만듭니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다.

In [ ]:
"""
MCP 클라이언트 실습: 터미널에 켜 둔 DART MCP 서버(우리가 만든 서버)와 Tavily 공식 MCP 서버(남이 만든 서버)에
연결해 LangGraph Agent가 두 서버의 도구를 함께 쓴다.
"""

import asyncio
import os
from typing import Annotated

import truststore
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm

truststore.inject_into_ssl()  # 사내망에서 Tavily(인터넷 HTTPS)에 연결할 때 회사 인증서를 믿게 한다
load_dotenv()


async def reachable(connection: dict) -> bool:
    """이 연결 설정으로 MCP 서버에 10초 안에 닿으면 True. 사내망에서 막힌 서버는 빼고 진행하려고 쓴다."""
    try:
        await asyncio.wait_for(
            MultiServerMCPClient({"check": connection}).get_tools(), timeout=10
        )
        return True
    except Exception:  # 연결 실패는 여러 오류가 한데 묶여(ExceptionGroup) 올라온다
        return False

### 5.1 연결할 서버 정하기

In [ ]:
# ── MCP 서버 연결 설정 ────────────────────────────────────────────────────────
# dart: 4장에서 터미널에 켜 둔 우리 DART 서버(SSE, 포트 8002)
MCP_CONFIG = {"dart": {"transport": "sse", "url": "http://localhost:8002/sse"}}

# tavily: Tavily가 인터넷에 운영하는 공식 MCP 서버(남이 만든 서버). HTTP로 접속하고, 키는 주소가 아니라
#         머리말(headers)에 넣는다(오류 메시지에 키가 찍히지 않게). 키가 있고 연결될 때만 더한다
TAVILY = {
    "transport": "streamable_http",
    "url": "https://mcp.tavily.com/mcp/",
    "headers": {"Authorization": f"Bearer {os.getenv('TAVILY_API_KEY', '').strip()}"},
}
if os.getenv("TAVILY_API_KEY", "").strip() and await reachable(TAVILY):
    MCP_CONFIG["tavily"] = TAVILY
print("사용할 MCP 서버:", ", ".join(MCP_CONFIG))
if "tavily" not in MCP_CONFIG:
    print(
        "Tavily는 뺐습니다(TAVILY_API_KEY가 없거나 Tavily 서버에 연결되지 않음). DART 도구만으로 진행합니다."
    )

### 5.2 도구 가져오기

In [ ]:
# MCP 설정으로 MultiServerMCPClient를 생성해 주세요.
client = MultiServerMCPClient(MCP_CONFIG)

# get_tools()는 서버에 연결해 도구 목록을 가져온다.
# MCP 서버에서 도구 목록을 가져와 주세요.
tools = await client.get_tools()
# Tavily 도구는 검색(tavily_search)만 쓴다. 나머지 넷(extract·crawl·map·research)은 크레딧을 많이 쓴다
tools = [
    t for t in tools if not t.name.startswith("tavily_") or t.name == "tavily_search"
]
print("=== 사용 가능한 MCP 도구 ===")
for t in tools:
    print(f"  - {t.name}: {t.description.strip().splitlines()[0][:70]}")
print()

### 5.3 Agent 그래프 만들기

In [ ]:
# 가져온 MCP 도구를 LLM과 ToolNode에 연결해 주세요.
llm = get_llm().bind_tools(tools)
tool_node = ToolNode(tools)


class State(TypedDict):
    messages: Annotated[list, add_messages]


def agent_node(state: State) -> dict:
    # State의 messages를 도구가 연결된 LLM에 전달해 주세요.
    return {"messages": [llm.invoke(state["messages"])]}


builder = StateGraph(State)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "agent")
# tools_condition으로 도구 호출 여부를 분기하고 tools 이후 agent로 돌아오게 연결해 주세요.
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

# 그래프를 컴파일해 주세요.
app = builder.compile()

### 5.4 질문하기

**✍️ CHANGE_HERE: 첫 질문의 회사를 바꾸면 DART 도구가 무엇을 돌려줄까요?**

**바꿔 볼 값** — 바로 아래 줄의 `삼성전자` 자리에 복사해 붙여 넣기 (하나씩 바꿔 실행)

서버의 회사 사전에 있는 회사
```text
카카오
```

서버의 회사 사전에 없는 회사
```text
쿠팡
```

값마다 이 셀을 다시 실행해 첫 질문의 결과를 봅니다. 다 봤으면 `삼성전자`로 되돌립니다.

In [ ]:
# 도구를 사용해 요청 처리
questions = [
    # ✍️ CHANGE_HERE: 첫 질문의 회사를 바꾸면 DART 도구가 무엇을 돌려줄까요?
    "삼성전자 최근 30일 공시를 알려 줘.",
    "SK하이닉스의 2025년 매출액과 영업이익은 얼마야?",
    "삼성전자 관련 최신 뉴스를 검색하고, DART 최근 공시와 함께 세 줄로 정리해 줘.",
]

for question in questions:
    mcp_graph_result = await app.ainvoke({"messages": [HumanMessage(content=question)]})
    results = mcp_graph_result["messages"]
    for m in results:
        m.pretty_print()
    print()

<details><summary>무슨 일이 생기나</summary>

`카카오`: 도구 호출의 인자가 `company: 카카오`로 바뀌고, Tool Message에 카카오의 최근 30일 공시(이번 실행에서는 6건: 기업설명회 개최 안내, 결산실적공시예고 등)가 나옵니다. 답도 그 목록을 정리합니다. 공시 건수와 제목은 실행하는 날에 따라 다릅니다.

`쿠팡`: 모델은 똑같이 `get_recent_disclosures`를 `company: 쿠팡`으로 부릅니다. 서버가 `'쿠팡'은(는) 이 서버의 회사 사전에 없습니다. 찾을 수 있는 회사: 삼성전자, SK하이닉스, …`를 돌려주자, 답은 목록에 없어 공시를 가져올 수 없다고 말하고 웹 검색을 제안했습니다(2026-10-21 사외 실행). 답은 도구가 돌려준 내용에 따라 정해집니다.

</details>

**볼 지점** (질문하기 셀)
- 질문마다 `Human Message` → `Ai Message`(도구 호출) → `Tool Message`(서버 도구의 결과) → `Ai Message`(답) 칸이 나옵니다.
  - 공시 질문: `get_recent_disclosures`가 가져온 삼성전자 최근 30일 공시(날짜·제목·제출인)
  - 실적 질문: `get_financial_summary`가 가져온 SK하이닉스 2025년 매출액 97.1조 원, 영업이익 47.2조 원
  - 뉴스 질문: `tavily_search`(뉴스)와 `get_recent_disclosures`(공시)를 함께 호출한 뒤 세 줄로 정리
- `[모의 DART]`로 시작하는 도구 결과가 보이면 실제 공시가 아닙니다. `.env`의 `DART_API_KEY`를 확인하고, 고쳤으면 4장의 서버를 다시 띄웁니다.
- Tavily를 뺀 상태라면 뉴스 질문에 검색 없이 공시만으로 정리하거나 "뉴스를 검색할 도구가 없다"고 답합니다. 정상입니다.
- 서버 터미널에는 DART 도구를 호출할 때마다 요청 기록이 늘어납니다. `tavily_search`는 인터넷의 Tavily 서버에서 실행되어 이 터미널에 기록이 남지 않습니다. 한 Agent가 두 곳의 서버 도구를 함께 쓴다는 뜻입니다.
- 답의 말투와 모양은 실행할 때마다 조금씩 다릅니다.

## 6. 정리

**이 노트북에서 한 것**
- FastMCP로 Tool·Resource·Prompt가 든 기본 서버를 읽고 터미널에서 띄운 뒤, `MultiServerMCPClient`로 연결해 도구 5개로 Agent 그래프를 만들었습니다. Resource와 Prompt는 도구 목록에 들어오지 않습니다.
- 연결 방식을 stdio로 바꾸면 노트북이 서버 파일을 직접 실행합니다. 호출할 때마다 서버가 새로 떠서 할 일 목록이 이어지지 않습니다.
- 실전 프로젝트에서는 DART 공시 API를 도구로 감싼 우리 서버와 Tavily가 운영하는 서버를 한 Agent에 함께 연결했습니다. 연결 설정만 바뀌고 Agent 코드는 같습니다.

실습이 끝나면 서버를 띄운 터미널(두 개)에서 **Ctrl+C**로 서버를 끕니다.